# rustymath

> Fast maths functions for f16, f32 and f64 in plain Rust.

rustymath gives you fast versions of the elementary maths functions, such as `exp`, `log`, `sin` and `pow`, for 16-, 32- and 64-bit floats. You can call them from Rust, where the compiler builds them into your own loops, or from Python, where they work on whole NumPy arrays. Most also take complex numbers. It also has special functions such as `gamma` and `erf`, probability distributions, APL's tolerant comparison, number theory and polynomials.

rustymath gives up the last bits of accuracy for speed. Every function is within 4 units in the last place (ulps) of the correctly rounded result. At 32 bits, most functions run two to six times as fast as the standard library's versions.

## Install

Install the Python package from PyPI. For Rust, see [Using rustymath from Rust](#using-rustymath-from-rust).

```sh
pip install rustymath
```

## Using rustymath from Python

Each function is a NumPy ufunc. It takes arrays of any shape, broadcasts them as NumPy does, and gives a result of the same width. Complex arrays work wherever the function has complex values, and integer arrays wherever NumPy's version takes them. `add`, `multiply`, `fmax` and `fmin` have `reduce` and `accumulate` methods that run rustymath's vectorized sums, products, maximums and minimums. Import rustymath beside NumPy:

In [ ]:
import numpy as np, rustymath as rm

Where NumPy has a function, rustymath uses the same name and arguments. To switch, you change only the import. Here `exp` keeps the 32-bit width of its argument:

In [ ]:
x = np.linspace(0, 2, 5, dtype=np.float32)
rm.exp(x)

array([1.       , 1.6487211, 2.718282 , 4.4816885, 7.389056 ],
      dtype=float32)

To check the accuracy yourself, compare with NumPy. `np.spacing` gives the size of one ulp at each value. Dividing the difference by it gives the distance between the two results in ulps:

In [ ]:
xs = np.linspace(-10, 10, 100_001)
y = np.exp(xs)
(np.abs(rm.exp(xs) - y) / np.spacing(y)).max()

np.float64(2.0)

Functions that NumPy lacks keep their own names. `root(n, y)` gives the `n`th root of `y`:

In [ ]:
rm.root(3, np.array([8., 27., 1000.]))

array([ 2.,  3., 10.])

The Rust names follow Rust's standard library instead. Python's `log`, `power` and `arcsin` are Rust's `ln`, `pow` and `asin`.

### Tolerant comparison

APL compares floats within a relative tolerance. Two floats are equal when their difference is at most the tolerance times the larger magnitude. With the usual tolerance, `0.1 + 0.2` equals `0.3`. `rm.equal` compares this way. Its tolerance `ct` defaults to `rm.CT`, 1e-14. Pass `ct=0` to compare exactly:

In [ ]:
rm.equal(0.1 + 0.2, 0.3), rm.equal(0.1 + 0.2, 0.3, ct=0)

(np.True_, np.False_)

`floor`, `ceil`, `remainder`, `gcd` and `lcm` take `ct` too. A value within tolerance of a whole number counts as that whole number:

In [ ]:
rm.floor(np.array([2.9999999999999996, 2.5]))

array([3., 2.])

`remainder` takes its arguments in NumPy's order. `rm.remainder(x, y)` is the Rust `residue(y, x)`.

`unique`, `index` and `isin` search within the same tolerance. `unique` and `isin` take the arguments of `numpy.unique` and `numpy.isin`. `index(x, y)` gives where in `x` each item of `y` first matches, or the size of `x` where nothing does:

In [ ]:
rm.unique([0.3, 1.0, 0.1 + 0.2]), rm.index([0.3, 1.0], 0.1 + 0.2)

(array([0.3, 1. ]), np.int64(0))

{{< include ../src/lib.md >}}

## Licence

Apache-2.0. Some algorithms and constants come from SLEEF and from Arm Optimized Routines. `THIRD-PARTY.md` has their notices.